# IFEval: canonical and corrective protocols through ScreamingFace

IFEval (arXiv:2311.07911) carries 541 prompts with machine-checkable constraints — word
counts, forbidden punctuation, required sections. The Engine grades every response with a
deterministic verifier: **no judge model, zero grading cost**.

The Engine publishes three independently named Benchmark protocols from the same IFEval
family:

- `ifeval`: the canonical single-answer protocol, comparable to published IFEval results.
- `ifeval-corrective`: three fixed answer/check attempts with sanitized verifier feedback.
- `ifeval-corrective-ensemble`: three members are separately checked and retried before a
  pinned Benchmark Judge selects one answer.

The SDK does not implement any protocol. It links a Model or Fusion into the complete
URL4 supplied by the selected Benchmark. Model calls are the only spend; discovery and
deterministic grading are free.

## Before running

The local AI Gateway must be running on `127.0.0.1:9105`, and the isolated Engine demo must be
running on `127.0.0.1:9108`. The connection panel sends the OpenRouter key through the Engine to
AI Gateway; the Client never calls AI Gateway directly.

For a host-local Engine, prepare IFEval's pinned cases (this also downloads the offline
NLTK tokenizer corpus the verifier reads) and pass the assets root explicitly:

```bash
uv run --with datasets python -m url4_cloud.benchmarks.ifeval.prepare \
  --out /tmp/screamingface-benchmark-assets/ifeval
URL4_BENCHMARK_ASSETS=/tmp/screamingface-benchmark-assets \
  uv run url4-cloud serve --local
```

`/opt/benchmarks` is the container image default and normally does not exist on the host.

In [ ]:
import screamingface as sf

## Connect OpenRouter

In [ ]:
sf.connect()

## Meet the benchmark

Before spending anything, read what the exams actually are. Discovery is free — plain
Engine REST, no model calls. The three resources share `family="ifeval"`, but have different
identities, revisions, protocols, costs, and scores.

In [ ]:
sf.benchmarks.list()

In [ ]:
canonical_benchmark = sf.benchmarks.get("ifeval")
corrective_benchmark = sf.benchmarks.get("ifeval-corrective")
ensemble_benchmark = sf.benchmarks.get("ifeval-corrective-ensemble")
canonical_benchmark, corrective_benchmark, ensemble_benchmark

### Read real prompts

Each prompt carries its constraints **in its own text** — "no commas", "at least 300
words", "highlight 3 sections". That is what makes IFEval machine-checkable: the Engine's
deterministic verifier re-reads the response against exactly those constraints, so
grading needs no judge model. Both variants use the same cases. Page further with
`canonical_benchmark.cases(limit=3, offset=100)`.

In [ ]:
canonical_benchmark.cases(limit=3)

## Define a Candidate

In [ ]:
haiku = sf.Model("openrouter/anthropic/claude-haiku-4.5")

## Run canonical IFEval

Canonical IFEval invokes the Candidate once and deterministically checks that answer.
`limit=3` selects three cases; it does not select a smaller Benchmark variant.

In [ ]:
canonical = sf.evaluate(
    haiku,
    benchmark="ifeval",
    limit=3,
    progress=False,
)
canonical

## Run the corrective IFEval variant

The corrective Benchmark owns a fixed three-attempt protocol. After attempts one and
two, its deterministic checker converts failures into sanitized constraint feedback and
the same Candidate tries again. URL4 currently has no conditional early stop, so all
three attempts run even if the first answer passes.

For `limit=3`, this means nine Candidate calls. The aggregate selects the earliest strict
pass, or the final attempt if none passes. Its result is a different experiment from
canonical IFEval and must be reported under its own Benchmark identity.

In [ ]:
corrective = sf.evaluate(
    haiku,
    benchmark="ifeval-corrective",
    limit=3,
    progress=False,
)
corrective

### Compare the two experiments

The corrective score may improve, but it pays for three attempts per case. Keep its score
and token use separate from canonical published IFEval results.

In [ ]:
{
    "canonical": {
        "score": canonical.candidates[0].score,
        "output_tokens": canonical.usage.output_tokens,
        "metrics": dict(canonical.candidates[0].metrics),
    },
    "corrective": {
        "score": corrective.candidates[0].score,
        "output_tokens": corrective.usage.output_tokens,
        "metrics": dict(corrective.candidates[0].metrics),
    },
}

### Audit the complete URL4

Each report contains the exact complete URL4 that executed. The canonical expression has
one Candidate invocation; the corrective expression has three and two feedback steps.
The client did not construct or interpret this workflow.

In [ ]:
canonical_url4 = canonical.candidates[0].url4
corrective_url4 = corrective.candidates[0].url4

print("canonical candidate invocations :", canonical_url4.count("/candidate?"))
print("corrective candidate invocations:", corrective_url4.count("/candidate?"))
print("corrective feedback steps       :", corrective_url4.count("!'feedback'"))

### Peek at the raw execution stream (optional)

Every run streams events while it executes. Today they describe raw url4 node
lifecycle (semantic events — "case 3, attempt 2" — are in flight engine-side), but
even the raw counts show the machine at work: one evaluation fans out into dozens of
nodes, and only the model calls cost anything.

In [ ]:
from collections import Counter

events = []
sf.evaluate(haiku, benchmark="ifeval-corrective", limit=1, on_event=events.append, progress=False)
Counter(event.kind for event in events)

## Evaluate a normal Fusion

A Fusion is still an ordinary Candidate: its members answer and its synthesizer produces
one final answer. Either Benchmark can invoke that Candidate without any IFEval-specific
SDK type.

Running a Fusion against `ifeval-corrective` retries and verifies the **Fusion's final
answer** three times. Kimi receives an explicit 16384-token ceiling because this reasoning
model can consume smaller completion budgets before emitting final answer text on longer
IFEval prompts. That is Candidate policy rather than Benchmark behavior.

In [ ]:
kimi = sf.Model(
    "openrouter/moonshotai/kimi-k2.6",
    params={"max_tokens": 16384},
)
deepseek = sf.Model("openrouter/deepseek/deepseek-v4-pro")
qwen = sf.Model("openrouter/qwen/qwen3.6-plus")

fusion = sf.Fusion(
    [kimi, deepseek, qwen],
    name="three-model fusion",
    synthesizer="openrouter/google/gemini-3-flash-preview",
)
fusion

In [ ]:
duel = sf.evaluate(
    [haiku, fusion],
    benchmark="ifeval-corrective",
    limit=1,
    progress=False,
)
duel

Both rows above use the same corrective Benchmark protocol. Their scores are
comparable to one another, while their token totals expose the cost of the Fusion. Candidate
lists execute concurrently, so this live example stays at one case to avoid turning the local
Gateway and upstream provider's concurrency limits into part of the experiment.

In [ ]:
{
    candidate.name: {
        "score": candidate.score,
        "output_tokens": candidate.usage.output_tokens,
    }
    for candidate in duel.candidates
}

## Run the member-level corrective ensemble

This is a separately revisioned experiment. It requires exactly three direct Model members.
The Benchmark invokes those members structurally, verifies and retries each one independently
for three attempts, and uses its pinned Flash Judge to select one answer per attempt. The
earliest passing selection becomes the final answer and receives canonical IFEval scoring.

The Fusion's ordinary final synthesizer is not invoked in this protocol. The SDK merely exposes
the three member expressions as universal bindings; all correction and selection behavior lives
inside the Engine-owned Benchmark URL4.

In [ ]:
ensemble_report = sf.evaluate(
    fusion,
    benchmark="ifeval-corrective-ensemble",
    limit=1,
    progress=False,
)
ensemble_report

In [ ]:
ensemble_url4 = ensemble_report.candidates[0].url4
print("member invocations per case:", ensemble_url4.count("/candidate_model_member_"))
print("judge invocations per case :", ensemble_url4.count("gemini-3-flash-preview"))

## Inspect the full Report

In [ ]:
corrective.candidates

In [ ]:
corrective.usage

In [ ]:
corrective.to_json()